# Un assistant qui répond à partir des documents de l'entreprise (RAG)

Un LLM ne connaît pas les documents internes d'une entreprise : interrogé sur eux, il invente une réponse plausible. La **génération augmentée par récupération** (RAG) lui fournit, avec la question, les extraits pertinents d'une base documentaire.

Ingrédients de cette démonstration, tous ouverts et exécutables sur Colab :

- un petit LLM « assistant », [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct)
- un modèle d'*embeddings* multilingue, qui transforme un texte en vecteur pour chercher par le sens

Un GPU accélère la démonstration (`Exécution > Modifier le type d'exécution`).

In [ ]:
!pip install -q "transformers>=4.56,<5" accelerate sentence-transformers

import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
LLM = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDER = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

tokenizer = AutoTokenizer.from_pretrained(LLM)
model = AutoModelForCausalLM.from_pretrained(LLM, dtype="auto").to(device)
embedder = SentenceTransformer(EMBEDDER, device=device)
print(device)

## La base documentaire

Le guide interne (fictif) de l'entreprise Acme : chaque paragraphe devient un extrait (*chunk*), préfixé par le titre de sa section.

In [ ]:
DOCUMENT = """
Télétravail
Les salariés en CDI peuvent télétravailler jusqu'à 2 jours par semaine, après accord de leur responsable.
La demande se fait dans l'outil RH « Kiosque », au moins 15 jours avant le premier jour de télétravail.
Acme fournit un écran et un siège ergonomique aux salariés qui télétravaillent au moins 1 jour par semaine.

Congés
Chaque salarié dispose de 25 jours de congés payés et de 11 jours de RTT par an.
Les congés de plus de 5 jours consécutifs se posent dans Kiosque au moins 1 mois à l'avance.

Notes de frais
Le repas lors d'un déplacement professionnel est remboursé dans la limite de 23,50 € par repas, sur justificatif.
Les notes de frais doivent être déposées dans Kiosque dans les 45 jours suivant la dépense.
Le remboursement intervient sur le salaire du mois suivant la validation.

Déplacements
Le train est obligatoire pour les trajets de moins de 4 h, en seconde classe.
Au-delà, l'avion en classe économique est autorisé, sur accord du directeur de département.
La nuit d'hôtel est remboursée jusqu'à 140 € à Paris et 95 € en province.

Sécurité informatique
L'authentification à deux facteurs est obligatoire sur tous les outils de l'entreprise.
Il est interdit de copier des données clients dans un outil d'IA public ; utiliser l'assistant interne « AcmeGPT ».
Tout courriel suspect doit être transféré à l'adresse securite@acme.example.

Formation
Chaque salarié a droit à 3 jours de formation par an, en plus de son CPF.
Les demandes de formation sont validées lors de l'entretien annuel.
"""

In [ ]:
chunks = []
for section in DOCUMENT.strip().split("\n\n"):
    title, *paragraphs = section.strip().splitlines()
    chunks.extend(f"{title} — {paragraph}" for paragraph in paragraphs)
print(len(chunks))
chunks[:3]

## Sans RAG : le modèle invente

Branchons le modèle comme assistant RH, sans lui donner le guide, qu'il n'a jamais vu :

In [ ]:
ROLE = "Tu es l'assistant RH d'Acme. Réponds aux questions des salariés de façon précise et concise."
QUESTION = "Quel est le plafond de remboursement d'un repas lors d'un déplacement ?"

def ask(messages: list[dict]) -> str:
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False,
                                temperature=None, top_p=None, top_k=None)
    prompt_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True)


print(ask([{"role": "system", "content": ROLE}, {"role": "user", "content": QUESTION}]))

La réponse a l'air sérieuse, mais elle est sans rapport avec le guide (qui dit 23,50 €) : c'est une **hallucination**. Selon la question, le modèle avoue parfois ne pas savoir, mais on ne peut pas compter dessus.

## Rechercher les extraits pertinents

On calcule le vecteur de chaque extrait une fois pour toutes. Pour une question, on garde les extraits dont le vecteur est le plus proche (produit scalaire de vecteurs normalisés, entre -1 et 1) :

In [ ]:
chunk_embeddings = embedder.encode(chunks, normalize_embeddings=True)


def retrieve(question: str, k: int = 3) -> list[tuple[float, str]]:
    question_embedding = embedder.encode(question, normalize_embeddings=True)
    scores = chunk_embeddings @ question_embedding
    best = scores.argsort()[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in best]


for score, chunk in retrieve(QUESTION):
    print(f"{score:.2f}  {chunk}")

## Avec RAG : répondre à partir des extraits

On donne au modèle les extraits trouvés, avec la consigne de s'y tenir et de citer ses sources :

In [ ]:
INSTRUCTIONS = ("Réponds uniquement à partir des extraits fournis, "
                "en citant le numéro de l'extrait utilisé. Si la réponse n'y figure pas, "
                "dis que tu ne sais pas.")
EXCERPTS_LABEL = "Extraits :"
QUESTION_LABEL = "Question :"

def rag(question: str) -> str:
    excerpts = "\n".join(f"[{i + 1}] {chunk}"
                         for i, (_, chunk) in enumerate(retrieve(question)))
    return ask([
        {"role": "system", "content": f"{ROLE} {INSTRUCTIONS}"},
        {"role": "user", "content": f"{EXCERPTS_LABEL}\n{excerpts}\n\n{QUESTION_LABEL} {question}"},
    ])


print(rag(QUESTION))

## Une question hors de la base

Le guide ne dit rien des voitures de fonction. Un bon assistant doit le reconnaître plutôt qu'inventer :

In [ ]:
OUTSIDE_QUESTION = "Quelle est la politique d'Acme sur les voitures de fonction ?"

print(rag(OUTSIDE_QUESTION))

## Une question formulée autrement

La recherche se fait par le sens, pas par les mots exacts : « déjeuner en mission » retrouve l'extrait sur le repas en déplacement.

In [ ]:
REPHRASED_QUESTION = "Combien puis-je dépenser pour déjeuner quand je suis en mission ?"

for score, chunk in retrieve(REPHRASED_QUESTION):
    print(f"{score:.2f}  {chunk}")
print()
print(rag(REPHRASED_QUESTION))

## À retenir

- Sans accès aux documents, un LLM hallucine avec aplomb
- Le RAG combine une recherche par le sens et une génération guidée par les extraits trouvés
- Citer les sources permet de vérifier les réponses
- La qualité dépend d'abord de la base documentaire (à jour, bien découpée) et de la recherche : une mauvaise recherche donne une mauvaise réponse
- Même avec le RAG, un petit modèle peut se tromper : il faut évaluer sur ses propres questions avant de déployer